# 实验 17 — dbt seeds：版本化的参考数据

**目标**：用 `dbt seed` 加载 CSV 作为 reference table。Seed 适合：

- 小而稳定的 mapping（货币-地区、国家-时区、test categories…）
- 数据由分析师在 PR 里维护，版本化在 git 里
- 不适合大表（>几万行）或频繁变化的数据

本仓库的 seed：[dbt_project/seeds/currency_regions.csv](../dbt_project/seeds/currency_regions.csv)

配置：[dbt_project/seeds/_seeds.yml](../dbt_project/seeds/_seeds.yml) —— 显式声明 column_types 和 tests，避免 dbt 推断错（`true/false` 推成 varchar 是常见坑）。

In [ ]:
import subprocess, os, duckdb
def dbt(*a):
    r = subprocess.run(['uv','run','dbt',*a], capture_output=True, text=True,
                       cwd='../dbt_project',
                       env={**os.environ,'DBT_PROFILES_DIR':'.'})
    return r.stdout + r.stderr

print(dbt('seed')[-1000:])

In [ ]:
con = duckdb.connect('../data/sandbox.duckdb')
print('=== seed 表 schema（注意 is_g10 是 boolean，因为 _seeds.yml 显式指定）===')
print(con.execute('describe main.currency_regions').df())
print()
con.execute('select * from main.currency_regions order by currency_code').df()

In [ ]:
# 在 model 里 ref() seed —— 跟 ref 别的 model 完全一样
# 演示：写一个临时 SQL 把 seed join 进 fct_daily_rates
sql = """
select f.currency, r.region, r.is_g10, count(*) as n
from main.fct_daily_rates f
left join main.currency_regions r on f.currency = r.currency_code
group by 1,2,3 order by 1
"""
con.execute(sql).df()

In [ ]:
# 改 CSV 后再跑 seed 是覆盖式的（write_disposition='replace'）
# dbt seed 默认 full-refresh；超过 1000 行的 seed 性能会有点慢——超过几万行就别用 seed 了
print(dbt('seed','--show')[-1500:])

## 生产环境踩坑提示

- **column_types 一定要显式声明**。`true/false` 默认会被 dbt-duckdb 当字符串读。日期、decimal 同理。
- **seed 在 build 顺序里默认靠前**：`dbt build` 先跑 seeds → snapshots → models → tests。
- **不要 seed 业务数据**。CSV 在 git 里 diff 还行（几百行），上千行就难审了。也不适合大数据量——dbt-duckdb 用 INSERT 一条一条插，慢。
- **生产替代**：参考数据如果由别的系统维护，更常见的是把它们也通过 dlt / Fivetran 拉进 raw 区，再用 dbt source 引用。Seed 更多用于“dbt 项目自家维护的小映射”。
- **测试 seed 也算 dbt test**：上面 yml 里加的 `not_null/unique` 跑 `dbt build` 时会执行。

## 思考题

- 把 seed 改成一行带 `is_g10=TRUE`（大写），再跑 seed 看会不会变成字符串 `'TRUE'`。
- 如果 seed 想作为下游 model 的 source（即不在 dbt 项目里维护，而是被 Fivetran 同步过来），dbt 里怎么写 source？为啥这种场景就不该用 seed？
- 改 CSV、再跑 seed，看 dbt 是把整表重建还是 upsert？